https://research.google/pubs/mapreduce-simplified-data-processing-on-large-clusters/

In [97]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import threading
from collections import defaultdict


%matplotlib inline

In [98]:
%%capture
! [ -f names.txt ] || wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt

In [99]:
df = pd.read_csv("names.txt", header=None)
names = df[0].str.lower()

assume each name line is a new document

In [113]:
membuff = [] # thread safe
def m_emit(key, value):
	membuff.append((key, value))

def mapper(key, value):
	for letter in value:
		m_emit(letter, 1)

In [114]:
gfs = defaultdict(dict)

def r_emit(pid, kv_value):
	k, v = kv_value
	gfs[f"part-{pid}"][k] = v

def reducer(pid, pairs):

	# boilerplate sort, group by key, loop over the groups handled by framework
	sort = defaultdict(list)
	for k,v in pairs:
		sort[k].append(v)

	# reduce begins now
	def reduce(key, value):
		r_emit(pid, (key, sum(value)))
	
	for k, v in sort.items():
		reduce(k, v)

In [115]:
N = 8 # map workers
R = 3 # reduce workers
MAX = 10

i = 0
while True:


	# document chunking processing
	if i+N < MAX:
		docs = zip(np.arange(i,i+N), names[i:i+N])
	else:
		docs = zip(np.arange(i,MAX), names[i:])

	# MAP: spawn workers, save onto memory
	workers = [threading.Thread(target=mapper, args=(k,v)) for k, v in docs]
	for w in workers: w.start()
	for w in workers: w.join()

	# SHUFFLE local disk memory buffer b4 getting pulled by reducer workers
	shuffled = defaultdict(list) # defaults to []
	for key , value in membuff:
		partition_id = hash(key) % R 	# all 'a' will go to same pid reducer
		shuffled[partition_id].append((key, value))
	

	# REDUCE workers.
	reducers = [threading.Thread(target=reducer, args=(rs)) for rs in shuffled.items()]
	for r in reducers: r.start()
	for r in reducers: r.join()

	i += N
	break

In [116]:
membuff[:10]

[('e', 1),
 ('m', 1),
 ('m', 1),
 ('a', 1),
 ('o', 1),
 ('l', 1),
 ('i', 1),
 ('v', 1),
 ('i', 1),
 ('a', 1)]

In [117]:
pid = 0
shuffled[pid][:10]

[('m', 1),
 ('m', 1),
 ('a', 1),
 ('a', 1),
 ('a', 1),
 ('a', 1),
 ('a', 1),
 ('a', 1),
 ('p', 1),
 ('a', 1)]

In [118]:
for pid, ls in shuffled.items():
	tmp = []
	for l, c in ls:
		if l not in tmp:
			tmp.append(l)

	print(f"reducer {pid}: {tmp}")

reducer 1: ['e', 'v', 'b', 'c', 't']
reducer 0: ['m', 'a', 'p']
reducer 2: ['o', 'l', 'i', 's', 'h', 'r']


In [119]:
gfs

defaultdict(dict,
            {'part-1': {'e': 4, 'v': 2, 'b': 1, 'c': 1, 't': 2},
             'part-0': {'m': 4, 'a': 11, 'p': 1},
             'part-2': {'o': 3, 'l': 5, 'i': 6, 's': 2, 'h': 2, 'r': 1}})